# Python Fundamentals Summary

**Stage 03 - Python Fundamentals**

This notebook uses deliberately fabricated toy return data to demonstrate Python, NumPy, pandas, and reusable project utilities. It does not make a market or trading claim.

## 1. Reproducible Imports

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display


def find_project_root() -> Path:
    """Locate project/ without assuming Jupyter's launch directory."""
    cwd = Path.cwd().resolve()
    for base in (cwd, *cwd.parents):
        for candidate in (base, base / 'project'):
            if (candidate / 'src' / 'utils.py').is_file():
                return candidate
    raise FileNotFoundError('Could not locate project/src/utils.py')


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import clean_column_names, get_group_summary, get_summary_stats

print('Python version:', sys.version.split()[0])
print('Project root:', PROJECT_ROOT)
print('NumPy version:', np.__version__)
print('pandas version:', pd.__version__)

Python version: 3.11.15
Project root: /Users/cengchengyu/Documents/NYU/Class/Boot Camp 4/CS HW/bootcamp_chengyu_zeng/project
NumPy version: 2.4.6
pandas version: 3.0.5


## 2. Toy Tabular Data and Column Cleaning

In [2]:
toy_data = pd.DataFrame(
    {
        'Trading Date': pd.date_range('2026-01-05', periods=8, freq='B'),
        'Asset Class': ['Equity', 'Bond', 'Equity', 'Bond', 'Equity', 'Bond', 'Equity', 'Bond'],
        'Daily Return': [0.011, -0.002, -0.006, 0.003, 0.008, -0.001, 0.004, 0.002],
    }
)

df = clean_column_names(toy_data)
assert list(df.columns) == ['trading_date', 'asset_class', 'daily_return']

df.info()
display(df.head())

<class 'pandas.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   trading_date  8 non-null      datetime64[us]
 1   asset_class   8 non-null      str           
 2   daily_return  8 non-null      float64       
dtypes: datetime64[us](1), float64(1), str(1)
memory usage: 364.0 bytes


,trading_date,asset_class,daily_return
0,2026-01-05,Equity,0.011
1,2026-01-06,Bond,-0.002
2,2026-01-07,Equity,-0.006
3,2026-01-08,Bond,0.003
4,2026-01-09,Equity,0.008


## 3. NumPy Vectorization

In [3]:
returns = df['daily_return'].to_numpy(dtype=float)
gross_returns = 1 + returns
cumulative_return = np.cumprod(gross_returns)[-1] - 1

assert np.allclose(gross_returns, returns + 1)
print('Vectorized gross returns:', gross_returns.round(4))
print(f'Toy cumulative return: {cumulative_return:.4%}')

Vectorized gross returns: [1.011 0.998 0.994 1.003 1.008 0.999 1.004 1.002]
Toy cumulative return: 1.9052%


## 4. Descriptive and Grouped Summaries

In [4]:
summary = get_summary_stats(df)
asset_class_summary = get_group_summary(
    df, group_col='asset_class', value_col='daily_return'
)

assert summary.loc[summary['feature'].eq('daily_return'), 'count'].iloc[0] == 8
assert set(asset_class_summary['asset_class']) == {'Bond', 'Equity'}

print('Numeric summary:')
display(summary)
print('Asset-class summary:')
display(asset_class_summary)

Numeric summary:


,feature,count,mean,std,min,25%,50%,75%,max
0,daily_return,8.0,0.002375,0.005476,-0.006,-0.00125,0.0025,0.005,0.011


Asset-class summary:


,asset_class,count,mean,std
0,Bond,4,0.00050,0.002380
1,Equity,4,0.00425,0.007411


## Notes

- The data are intentionally fabricated and are only for code practice.
- `src/utils.py` contains reusable column-cleaning and summary helpers for later data stages.
- Run **Restart & Run All** before committing to confirm the notebook has no hidden state.